In [1]:
from datasets import load_dataset
from PIL import Image
from torchvision import transforms
import torch
from torch.utils.data import DataLoader


dataset = load_dataset("nielsr/CelebA-faces", split="train", streaming=False)

/Users/isabelvalladolid/Documents/DeepLearningProjects/autoencoder-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

In [3]:
for i, example in enumerate(dataset.take(5)):
    image = example["image"]
    print(f"Ejemplo {i}: Shape {image.size}, Mode {image.mode}")
    image.show()

Ejemplo 0: Shape (178, 218), Mode RGB
Ejemplo 1: Shape (178, 218), Mode RGB
Ejemplo 2: Shape (178, 218), Mode RGB
Ejemplo 3: Shape (178, 218), Mode RGB
Ejemplo 4: Shape (178, 218), Mode RGB


In [4]:
dataset

Dataset({
    features: ['image'],
    num_rows: 202599
})

In [5]:
_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5], 
        std=[0.5, 0.5, 0.5]
    )
])

In [6]:
def transform_fn(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples

dataset.set_transform(transform_fn)

In [7]:
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    return {"pixel_values": pixel_values}

train_dataloader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=True, 
    collate_fn=collate_fn
)

In [22]:
import keras 
from keras import layers, backend
from keras.models import Model


def create_encoder():
    
    inputs = keras.Input(shape=(64,64,3))

    # 32x4x4 conv, stride 2 + BN + LeakyReLU
    x = layers.Conv2D(32,(4, 4), strides=2, padding='same', activation=None)(inputs)
    x = layers.BatchNormalization(axis=-1)(x) # axis=-1 as data format is 'channels_last' and we want to normalize each channel
    x = layers.LeakyReLU(alpha=0.3)(x)

    #64x4x4 conv, stride 2 + BN + LeakyReLU
    x = layers.Conv2D(64,(4, 4), strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    #128x4x4 conv, stride 2 + BN + LeakyReLU
    x = layers.Conv2D(128,(4, 4), strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    #256x4x4 conv, stride 2 + BN + LeakyReLU
    x = layers.Conv2D(256,(4, 4), strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # flatten the output
    x = layers.Flatten(data_format='channels_last')(x)

    # distribution parameters
    z_mean = layers.Dense(units=100, activation=None)(x)
    z_log_sigma = layers.Dense(units=100, activation=None)(x)

    # use these parameters to sample new similar points from the latent space:
    def sampling(args):
        z_mean, z_log_sigma = args
        epsilon = backend.random_normal(shape=(1, 100), mean=0., stddev=0.1)
        #epsilon = backend.random_normal(shape=(1, 100), mean=0., stddev=1.)
        return z_mean + backend.exp(z_log_sigma / 2) * epsilon 

    z = layers.Lambda(sampling)([z_mean, z_log_sigma])

    # Create encoder
    encoder = Model(inputs,[z_mean, z_log_sigma, z], name='encoder')
    encoder.summary()
    
    return encoder


def create_decoder():
    
    inputs = keras.Input(shape=(100))

    x = layers.Dense(units=4096, activation='relu')(inputs)
    x = layers.Reshape((4,4,256))(x)

    # upsample + conv128x3x3 + BN + LeakyReLU
    x = layers.UpSampling2D((2,2), data_format='channels_last', interpolation='nearest')(x) 
    x = layers.Conv2D(128,(3, 3), padding='same', strides=(1,1))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # upsample + conv64x3x3 + BN + LeakyReLU
    x = layers.UpSampling2D((2,2))(x) 
    x = layers.Conv2D(64,(3, 3), padding='same')(x) 
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # upsample + conv32x3x3 + BN + LeakyReLU
    x = layers.UpSampling2D((2,2))(x) 
    x = layers.Conv2D(32,(3, 3), padding='same')(x) 
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # upsample + conv128x3x3
    x = layers.UpSampling2D((2,2))(x) 
    decoded = layers.Conv2D(3,(3, 3), padding='same')(x)

    decoder = Model(inputs, decoded, name='decoder')
    return decoder

In [23]:
class PVAE(keras.Model):
    
    def __init__(self, preprocesser, encoder, decoder, alpha, beta, **kwargs):
        super(PVAE, self).__init__(**kwargs)
        self.preprocesser = preprocesser
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.loss_wo_weight_tracker = keras.metrics.Mean(name="loss_wo_weight")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.alpha = alpha
        self.beta = beta

    @property
    def metrics(self):
        # We list our `Metric` objects here so that `reset_states()` can be
        # called automatically at the start of each epoch
        # or at the start of `evaluate()`.
        # If you don't implement this property, you have to call
        # `reset_states()` yourself at the time of your choosing.
        return [
            self.loss_tracker,
            self.loss_wo_weight_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]
    
    def forward(self, data):
        # Forward pass
        preprocessed = self.preprocesser(data)
        z_mean, z_log_sigma, z = self.encoder(preprocessed)
        reconstruction = self.decoder(z)
        return z_mean, z_log_sigma,reconstruction
    
    def compute_loss(self, data, z_mean, z_log_sigma, reconstruction):
        reconstruction_loss = tf.reduce_mean(keras.losses.mean_squared_error(data, reconstruction))
        kl_loss = -0.5 * (1 + z_log_sigma - tf.square(z_mean) - tf.exp(z_log_sigma))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        loss_wo_weight = reconstruction_loss + kl_loss
        loss = self.alpha * reconstruction_loss + self.beta * kl_loss
        return reconstruction_loss, kl_loss, loss, loss_wo_weight
    
    def train_step(self, data):
        with tf.GradientTape() as tape:
            # Forward pass
            z_mean, z_log_sigma, reconstruction = self.forward(data)
            # Compute the loss value
            reconstruction_loss, kl_loss, loss, loss_wo_weight = self.compute_loss(data, z_mean, z_log_sigma, reconstruction)
        # Compute gradients
        grads = tape.gradient(loss, self.trainable_variables)
        # Update weights
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        # Update metrics (metrics = losses in our case)
        self.loss_tracker.update_state(loss)
        self.loss_wo_weight_tracker.update_state(loss_wo_weight)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        # Return metrics
        return {
            "loss": self.loss_tracker.result(),
            "loss_wo_weight": self.loss_wo_weight_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }
    
    def call(self, data):
        # Forward pass
        z_mean, z_log_sigma, reconstruction = self.forward(data)
        # Compute the loss value
        reconstruction_loss, kl_loss, loss, loss_wo_weight = self.compute_loss(data, z_mean, z_log_sigma, reconstruction)
        # Update metrics (metrics = losses in our case)
        self.loss_tracker.update_state(loss)
        self.loss_wo_weight_tracker.update_state(loss_wo_weight)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        # Return metrics
        return {
            "loss": self.loss_tracker.result(),
            "loss_wo_weight": self.loss_wo_weight_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
            "reconstruction":reconstruction,
        }        

In [25]:
def create_preprocesser():
    
    inputs = keras.Input(shape=(64,64,3))
    
    preprocessed = layers.Rescaling(1./255)(inputs)
    
    preprocesser = Model(inputs,preprocessed, name='preprocesser')
    
    return preprocesser

In [ ]:
from keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

alpha = 1.8
beta = 10
lr = 0.0015

ppreprocesser = create_preprocesser()
pencoder = create_encoder()
pdecoder = create_decoder()
pvae = PVAE(ppreprocesser, pencoder, pdecoder, alpha, beta)
pvae.compile(optimizer=keras.optimizers.Adam(lr))

callbacks = [
    ReduceLROnPlateau(monitor='val_loss_wo_weight', factor=0.2, patience=2, verbose=1, mode='min'),
    ModelCheckpoint(filepath='./pvae/', monitor="val_loss_wo_weight", verbose=1, save_best_only=True)
]
history_pvae = pvae.fit(x=train_ds, epochs=30, validation_data=val_ds, batch_size=128, callbacks=callbacks)

In [17]:
print(f"Total de batches: {len(train_dataloader)}")

images_array = []

for epoch in range(1):
    for batch_idx, batch in enumerate(train_dataloader):
        images = batch["pixel_values"] # Shape: [64, 3, 64, 64]
        images_array.append(images.numpy())
        print(images_array)
        print(batch_idx)

Total de batches: 6332
[array([[[[-0.75686276, -0.81960785, -0.9137255 , ..., -0.84313726,
          -0.79607844, -0.8039216 ],
         [-0.7490196 , -0.8117647 , -0.90588236, ..., -0.84313726,
          -0.79607844, -0.8039216 ],
         [-0.7490196 , -0.8039216 , -0.8980392 , ..., -0.8509804 ,
          -0.8039216 , -0.8039216 ],
         ...,
         [-0.52156866, -0.52156866, -0.52156866, ...,  0.30980396,
          -0.69411767, -0.8901961 ],
         [-0.52156866, -0.52156866, -0.52156866, ...,  0.3411765 ,
          -0.6313726 , -0.8980392 ],
         [-0.52156866, -0.52156866, -0.52156866, ...,  0.39607847,
          -0.54509807, -0.8901961 ]],

        [[-0.7254902 , -0.79607844, -0.90588236, ..., -0.8352941 ,
          -0.78039217, -0.77254903],
         [-0.7176471 , -0.7882353 , -0.8980392 , ..., -0.8352941 ,
          -0.78039217, -0.77254903],
         [-0.7176471 , -0.78039217, -0.8901961 , ..., -0.84313726,
          -0.7882353 , -0.77254903],
         ...,
         [